In [1]:
import os

print(os.listdir("/kaggle/input"))

['datasets']


In [2]:
import os

print(os.listdir("/kaggle/input/datasets"))

['harshitabansal307']


In [3]:
import os

print(os.listdir("/kaggle/input/datasets/harshitabansal307"))

['deepfashion-pruned-dataset']


In [4]:
import os

BASE_PATH = "/kaggle/input/datasets/harshitabansal307/deepfashion-pruned-dataset"
print(os.listdir(BASE_PATH))

['validation', 'test', 'train']


## Step 1: Import Required Libraries

In this cell, we import all necessary libraries for:
- File handling
- JSON parsing
- PyTorch model development
- Image transformations
- Evaluation metrics

In [5]:
import os
import json
from tqdm import tqdm

import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.optim as optim

from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models

from PIL import Image

from sklearn.metrics import classification_report, f1_score, precision_score, recall_score, roc_auc_score, roc_curve

## Step 2: Define Dataset Paths

Here we define paths to:
- Train images and annotations
- Validation images and annotations
- Test images

These paths point to the pruned DeepFashion2 dataset uploaded on Kaggle.

In [7]:
BASE_PATH = "/kaggle/input/datasets/harshitabansal307/deepfashion-pruned-dataset"

TRAIN_IMG_DIR = os.path.join(BASE_PATH, "train", "train", "images")
TRAIN_ANNO_DIR = os.path.join(BASE_PATH, "train", "train", "annos")

VAL_IMG_DIR = os.path.join(BASE_PATH, "validation", "validation", "images")
VAL_ANNO_DIR = os.path.join(BASE_PATH, "validation", "validation", "annos")

TEST_IMG_DIR = os.path.join(BASE_PATH, "test", "test", "images")

print("Train images:", len(os.listdir(TRAIN_IMG_DIR)))
print("Train annos:", len(os.listdir(TRAIN_ANNO_DIR)))
print("Val images:", len(os.listdir(VAL_IMG_DIR)))
print("Val annos:", len(os.listdir(VAL_ANNO_DIR)))
print("Test images:", len(os.listdir(TEST_IMG_DIR)))

Train images: 144174
Train annos: 144174
Val images: 23741
Val annos: 23741
Test images: 62629


## Step 3: Define Category Mapping (Frequency Order)

We use the top-5 categories in descending order of frequency:

[1, 8, 7, 2, 9]

This order will define the class index positions (0–4).

In [8]:
TOP5 = [1, 8, 7, 2, 9]

category_map = {cid: idx for idx, cid in enumerate(TOP5)}

NUM_CLASSES = 5

print("Category mapping:")
for cid, idx in category_map.items():
    print(f"Original ID {cid} → New Index {idx}")

Category mapping:
Original ID 1 → New Index 0
Original ID 8 → New Index 1
Original ID 7 → New Index 2
Original ID 2 → New Index 3
Original ID 9 → New Index 4


## Step 4: Parse JSON Files and Create Multi-Label Targets

Each image may contain multiple clothing items.

For classification, we convert the category_ids in each annotation
into a multi-label vector of size 5.

Class order (frequency order):
[1, 8, 7, 2, 9]

Example:
If an image contains categories 1 and 9,
label vector becomes:
[1, 0, 0, 0, 1]

In [9]:
def create_samples(img_dir, anno_dir):
    samples = []
    
    for anno_file in tqdm(os.listdir(anno_dir)):
        
        if not anno_file.endswith(".json"):
            continue
        
        img_id = anno_file.replace(".json", "")
        img_path = os.path.join(img_dir, img_id + ".jpg")
        anno_path = os.path.join(anno_dir, anno_file)
        
        if not os.path.exists(img_path):
            continue
        
        with open(anno_path, "r") as f:
            data = json.load(f)
        
        # initialize multi-label vector
        label_vector = [0] * NUM_CLASSES
        
        # each annotation has {"items": [...]}
        for item in data["items"]:
            cid = item["category_id"]
            
            if cid in category_map:
                mapped_idx = category_map[cid]
                label_vector[mapped_idx] = 1
        
        samples.append((img_path, label_vector))
    
    return samples


print("Creating train samples...")
train_samples = create_samples(TRAIN_IMG_DIR, TRAIN_ANNO_DIR)

print("Creating validation samples...")
val_samples = create_samples(VAL_IMG_DIR, VAL_ANNO_DIR)

print("Total train samples:", len(train_samples))
print("Total val samples:", len(val_samples))

Creating train samples...


100%|██████████| 144174/144174 [15:36<00:00, 153.89it/s]


Creating validation samples...


100%|██████████| 23741/23741 [02:58<00:00, 133.20it/s]

Total train samples: 144174
Total val samples: 23741


## Step 5: Compute Class Imbalance Weights

DeepFashion2 is imbalanced — some clothing categories appear
much more frequently than others.

To prevent the model from ignoring less frequent classes,
we compute class-wise positive weights.

These weights will be used in:
BCEWithLogitsLoss(pos_weight=...)

This improves macro-F1 performance.

In [10]:
# Count positive samples per class
class_counts = np.zeros(NUM_CLASSES)

for _, label in train_samples:
    class_counts += np.array(label)

print("Positive samples per class:")
for cid, idx in category_map.items():
    print(f"Category {cid} → Count: {int(class_counts[idx])}")


# Total samples
total_samples = len(train_samples)

# Compute positive weights
# Formula: (N - P) / P
pos_weights = (total_samples - class_counts) / class_counts

print("\nComputed pos_weight for BCE:")
print(pos_weights)

# Convert to torch tensor
pos_weights = torch.tensor(pos_weights, dtype=torch.float32)

Positive samples per class:
Category 1 → Count: 70586
Category 8 → Count: 54969
Category 7 → Count: 36332
Category 2 → Count: 35751
Category 9 → Count: 30625

Computed pos_weight for BCE:
[1.04252968 1.62282377 2.96823737 3.03272636 3.70772245]


## Step 6: Define Image Transformations

CNN models require fixed-size, normalized inputs.

We apply:

For Training:
- Resize to 224×224
- Random horizontal flip
- Small rotation
- Color jitter
- Normalize using ImageNet mean & std

For Validation/Test:
- Resize
- Normalize
(No augmentation)

In [11]:
from torchvision import transforms

# ImageNet normalization values (important for transfer learning)
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]


# Train transforms (with augmentation)
train_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(10),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD)
])


# Validation/Test transforms (no augmentation)
val_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD)
])

print("Transforms defined successfully.")

Transforms defined successfully.


## Step 7: Create Custom PyTorch Dataset Class

We create a custom Dataset class that:

- Loads images using PIL
- Applies transforms
- Returns:
    image tensor
    multi-label tensor

This allows efficient batching using DataLoader.

In [22]:
class DeepFashionMultiLabelDataset(Dataset):
    
    def __init__(self, samples, transform=None):
        """
        samples: list of (image_path, label_vector)
        transform: torchvision transforms
        """
        self.samples = samples
        self.transform = transform
    
    def __len__(self):
        return len(self.samples)
    
    def __getitem__(self, idx):
        img_path, label = self.samples[idx]
        
        # Load image
        image = Image.open(img_path).convert("RGB")
        
        # Apply transforms
        if self.transform:
            image = self.transform(image)
        
        if label is None:
            return image
        else:
            label = torch.tensor(label, dtype=torch.float32)
            return image, label

## Step 8: Create Train and Validation Dataset Objects

In [15]:
train_dataset = DeepFashionMultiLabelDataset(
    samples=train_samples,
    transform=train_transforms
)

val_dataset = DeepFashionMultiLabelDataset(
    samples=val_samples,
    transform=val_transforms
)

print("Train dataset size:", len(train_dataset))
print("Validation dataset size:", len(val_dataset))

Train dataset size: 144174
Validation dataset size: 23741


## Step 9: Create DataLoaders

DataLoader allows batching and shuffling.

In [16]:
BATCH_SIZE = 32

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=2,
    pin_memory=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

print("DataLoaders created.")

DataLoaders created.


In [17]:
images, labels = next(iter(train_loader))

print("Image batch shape:", images.shape)
print("Label batch shape:", labels.shape)

Image batch shape: torch.Size([32, 3, 224, 224])
Label batch shape: torch.Size([32, 5])


## Create Test Samples (Images Only)

Test split does not contain annotations.
We only store image paths.

In [23]:
# Create test samples (image path, dummy label)
test_samples = []

for img_file in os.listdir(TEST_IMG_DIR):
    if img_file.endswith(".jpg"):
        img_path = os.path.join(TEST_IMG_DIR, img_file)
        test_samples.append((img_path, None))

print("Total test samples:", len(test_samples))

Total test samples: 62629


## Create Test Dataset

We use validation transforms (no augmentation).

In [24]:
test_dataset = DeepFashionMultiLabelDataset(
    samples=test_samples,
    transform=val_transforms
)

print("Test dataset created.")

Test dataset created.


## Save Preprocessed Data for Reuse

We save:
- train_samples
- val_samples
- pos_weights
- category_map

These will be loaded in separate model training notebooks.

In [25]:
import torch
import os

SAVE_DIR = "/kaggle/working/preprocessed"
os.makedirs(SAVE_DIR, exist_ok=True)

torch.save(train_samples, os.path.join(SAVE_DIR, "train_samples.pt"))
torch.save(val_samples, os.path.join(SAVE_DIR, "val_samples.pt"))
torch.save(pos_weights, os.path.join(SAVE_DIR, "pos_weights.pt"))
torch.save(category_map, os.path.join(SAVE_DIR, "category_map.pt"))

print("Preprocessed files saved successfully.")

Preprocessed files saved successfully.
